In [2]:
import numpy as np

X = np.array([
    [1, 2],
    [2, 1],
    [3, 4],
    [4, 3]
])

y = np.array([3, 6, 5, 8])

ones = np.ones(X.shape[0])

X_b = np.column_stack((ones, X))

print(X_b)

[[1. 1. 2.]
 [1. 2. 1.]
 [1. 3. 4.]
 [1. 4. 3.]]


In [3]:
A = X_b.T @ X_b
c = X_b.T @ y
theta = np.linalg.solve(A, c)
print(theta)

[ 3.  2. -1.]


In [4]:
y_pred = X_b @ theta
print(y_pred)
assert np.allclose(y_pred, y)

[3. 6. 5. 8.]


In [5]:
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X_b = np.column_stack((ones, X))
    A = X_b.T @ X_b
    c = X_b.T @ y
    theta = np.linalg.solve(A, c)
    return theta

theta_test = train_linear_regression(X, y)

print(theta_test)

assert np.allclose(theta_test, [3, 2, -1])


[ 3.  2. -1.]


In [6]:
from sklearn.linear_model import LinearRegression

In [7]:
model = LinearRegression()
model.fit(X, y)
print(model.intercept_, model.coef_)
scikit_pred = model.predict(X)

assert np.allclose(scikit_pred, y_pred)

2.9999999999999996 [ 2. -1.]


In [8]:
import pandas as pd
df = pd.read_csv('../../01-intro/1.9 Bike Sharing/data/day.csv')
df['dteday'] = pd.to_datetime(df['dteday'])
df = df.sort_values('dteday')
X_calendar = df[["season","yr","mnth","holiday","weekday","workingday"]]
print(X_calendar.head())

y = df["cnt"]
X_train = X_calendar.iloc[:511]
X_val = X_calendar.iloc[511:511+109]
X_test = X_calendar.iloc[511+109:]
y_train = df["cnt"].iloc[:511]
y_val = df["cnt"].iloc[511:511+109]



   season  yr  mnth  holiday  weekday  workingday
0       1   0     1        0        6           0
1       1   0     1        0        0           0
2       1   0     1        0        1           1
3       1   0     1        0        2           1
4       1   0     1        0        3           1


In [9]:
print(X_train.shape, X_val.shape, X_test.shape, y_train.shape, y_val.shape)

assert X_train.shape[0] == 511
assert X_val.shape[0] == 109
assert X_test.shape[0] == 111

(511, 6) (109, 6) (111, 6) (511,) (109,)


In [10]:
theta_bike = train_linear_regression(X_train, y_train)
print(theta_bike)
print(theta_bike.shape)

assert theta_bike.shape[0] == X_train.shape[1] + 1

[1257.75176933  727.13785696 2205.16933806   23.37142286 -524.43608454
   39.96214092  106.27631802]
(7,)


In [11]:
X_train_b = np.column_stack((np.ones(X_train.shape[0]), X_train))
X_val_b = np.column_stack((np.ones(X_val.shape[0]), X_val))
y_train_pred = X_train_b @ theta_bike
y_val_pred = X_val_b @ theta_bike

print(y_val_pred[:5])
assert y_val_pred.shape[0] == y_val.shape[0]

[5273.82678116 5034.05393561 4549.579992   5220.25453548 5260.21667641]


In [12]:
assert y_train_pred.shape[0] == y_train.shape[0]

train_mae = np.abs(y_train - y_train_pred).mean()
val_mae = np.abs(y_val - y_val_pred).mean()
print(train_mae, val_mae)


1022.1137607602232 1124.77840589107


In [13]:
new_model = LinearRegression()
new_model.fit(X_train, y_train)
new_model_pred = new_model.predict(X_val)
scikit_mae = np.abs(y_val - new_model_pred).mean()
print(scikit_mae)

assert np.allclose(scikit_mae, val_mae)

1124.7784058910688


In [14]:
assert np.allclose(new_model_pred, y_val_pred)

abs_difference = np.abs(new_model_pred - y_val_pred).max()
print(abs_difference)

4.547473508864641e-12


## Day 11 — Linear Regression Experiment Results

### 1. Experiment Setup

**Objective:** Train a linear regression model to predict tomorrow's total daily bike rentals and compare a custom NumPy implementation with scikit-learn's `LinearRegression`.

**Dataset:** UCI Bike Sharing daily dataset, containing 731 observations from 2011–2012.

**Target:** `cnt` — total daily bike rentals.

**Features used:**
- `season` — Season of the year
- `yr` — Year (0 = 2011, 1 = 2012)
- `mnth` — Month
- `holiday` — Whether the day is a holiday
- `weekday` — Day of the week
- `workingday` — Whether the day is a working day

Features such as `casual` and `registered` were excluded because their sum equals the target variable, creating direct target leakage. Actual weather measurements were also excluded because they would not be available when making predictions the day before.

**Chronological split:**

| Dataset | Observations | Purpose |
|---|---:|---|
| Training | 511 | Learn regression coefficients |
| Validation | 109 | Evaluate generalization and compare models |
| Test | 111 | Reserved for final evaluation |

The data was sorted chronologically before splitting. Both regression implementations used identical training data and features. Neither model was trained using validation or test targets.

### 2. Model Performance

| Model | Training MAE | Validation MAE |
|---|---:|---:|
| Training-mean baseline | — | 2,950.82 |
| Seasonal baseline | — | 2,235.47 |
| NumPy Linear Regression | 1,022.11 | 1,124.78 |
| scikit-learn LinearRegression | — | 1,124.78 |

The NumPy linear regression achieved approximately **49.7% lower validation MAE** than the seasonal baseline.

The difference between NumPy training MAE and validation MAE was approximately 102.66 rentals, indicating a relatively modest generalization gap on this split.

The NumPy and scikit-learn models produced essentially identical validation predictions.

**Maximum absolute prediction difference:** `4.547473508864641e-12`

An `np.allclose()` assertion comparing all 109 validation predictions passed.

These results demonstrate consistent numerical behavior across the two implementations on this dataset. They do not guarantee the same performance on future or substantially different data.

### 3. Implementation Differences

**NumPy implementation:**

I implemented ordinary least squares using the normal equations. My function automatically adds a column of ones to represent the intercept, computes the matrix products, and uses `np.linalg.solve()` to obtain the coefficients.

This implementation helped me understand how regression coefficients are learned from features and targets rather than manually specified.

**scikit-learn implementation:**

I used `LinearRegression().fit(X_train, y_train)` to train the model. Scikit-learn automatically handles the intercept by default and uses a numerically robust least-squares solver for dense data.

Both implementations minimize the sum of squared training residuals, although I evaluated their performance using MAE.

One limitation of the NumPy implementation is that solving the normal equations can be numerically unstable when features are highly correlated. If the feature matrix contains redundant columns, the matrix may become singular and `np.linalg.solve()` may raise a `LinAlgError`.

Scikit-learn's least-squares approach can handle rank-deficient feature matrices without requiring a unique solution.

For production-oriented implementations, `np.linalg.lstsq()` or a well-tested library such as scikit-learn is generally preferable to explicitly solving the normal equations.

### 4. Limitations and Deployment Considerations

**Historical data and distribution shift:** The dataset only contains observations from 2011–2012. Rental patterns may change because of infrastructure, weather patterns, consumer behavior, or other external factors. Strong historical validation results do not guarantee future accuracy.

**Year encoding:** The `yr` feature is binary and represents only 2011 and 2012. The model learned a positive coefficient of approximately 2,205 for this feature, but that does not establish a recurring annual increase or support reliable extrapolation to later years.

**Categorical feature encoding:** Features such as season, month, and weekday are treated as numeric values. This assumes a linear relationship between their encoded values and predicted rentals. One-hot encoding or cyclical representations may better capture their actual relationships.

**Missing weather information:** The model uses only calendar features. It cannot directly account for unexpected rain, temperature changes, or other weather conditions that influence rental demand. Future improvements could incorporate weather forecasts that are genuinely available before prediction time.

**Model limitations:** Ordinary linear regression cannot naturally represent complex nonlinear interactions and may produce negative rental predictions, even though actual counts cannot be negative.

**Test evaluation:** The test dataset remains untouched. Validation performance was used to compare modeling approaches, but final test evaluation is still required after model selection.

### 5. Conclusion

I successfully implemented linear regression training from scratch using NumPy and compared it against scikit-learn on identical features and training observations.

Both models achieved a validation MAE of approximately 1,124.78 rentals, substantially outperforming the previous seasonal baseline of 2,235.47.

The extremely small prediction difference demonstrates numerical agreement between the two implementations for this experiment.

**Key takeaway:** Linear regression training involves finding coefficients that minimize an objective function, while reliable ML engineering also requires appropriate data splitting, leakage prevention, numerical validation, baseline comparisons, and careful interpretation of generalization performance.

The notebook was restarted and executed successfully from beginning to end.